### Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torch.optim import Optimizer
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import wandb

In [ ]:
torch.manual_seed(42)

In [ ]:
PROJECT_NAME = "recursive-synaptic-balance"

default_config = {
    "epochs": 5,
    "optimizer": "rsb",
    "dataset": "MNIST",
    "batch_size": 1,
    "lr": 0.01,
    "alpha": 0.01,
    "beta": 0.01,
    "input_dim": -1,
    "hidden_dim": -1,
    "output_dim": -1,
}

# TODO: Implement
sweep_config = {
    "method": "grid",
    "metric": {
        "name": "train_loss",
        "goal": "minimize"
    },
    "parameters": {
        "learning_rate": {
            "values": [0.1]
        },
        "batch_size": {
            "values": [16]
        },
        "regression_penalty": {
            "values": [0.0]
        }
    }
}

### Datasets

In [ ]:
mnist_transform = transforms.Compose([ # It would be nice to check these numbers, if we have time
    transforms.ToTensor(), 
    transforms.Normalize((0.1307,), (0.3081,))
])
cifar10_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))
])
cifar100_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

### Models

In [ ]:
class CNN(nn.Module):
    def __init__(self, input_size, in_channels=1, num_classes=10):
        super(CNN, self).__init__()
        self.maxpool = nn.MaxPool2d(2)
        self.relu    = nn.ReLU()
        
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 8,  kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(8,  32, kernel_size=3, padding=1)

        h, w = input_size
        self.fc1 = nn.Linear(32 * (h // 4) * (w // 4), 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.maxpool(self.relu(self.conv1(x)))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = self.maxpool(self.relu(self.conv4(x)))

        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [ ]:
class FNN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(FNN, self).__init__()
        self.relu = nn.ReLU()
        
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, output_dim)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return F.log_softmax(x, dim=1)

### Criterion

In [ ]:
class RSB_Loss(nn.Module):
    def __init__(self, alpha=0.01, beta=0.005):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta

    def forward(self, outputs, targets, model):
        data_loss = F.nll_loss(outputs, targets)

        weights = [p for p in model.parameters() if len(p.shape) > 1]
        norms   = [torch.norm(w)**2 for w in weights]

        l2_loss = self.alpha * norms[-1]
        balance_terms = [(norms[i] - norms[i+1])**2 for i in range(len(norms) - 1)]
        balance_loss = self.beta * torch.stack(balance_terms).sum() if balance_terms else norms[-1].new_zeros(1)
        regr_loss = l2_loss + balance_loss

        return data_loss + regr_loss

### Optimizer

In [ ]:
class RSB(Optimizer):
    def __init__(
        self, 
        params, 
        lr,
        alpha,
        beta,
    ):
        defaults = dict(lr=lr, alpha=alpha, beta=beta)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None

        for group in self.param_groups:
            lr    = group["lr"]
            alpha = group["alpha"]
            beta  = group["beta"]

            # 1. Apply gradient step to all params
            for p in group["params"]:
                if p.grad is not None:
                    p.add_(p.grad, alpha=-lr)

            # 2. Apply balance/L2 prior only to weight matrices
            weights = [p for p in group["params"] if p.grad is not None and len(p.shape) > 1]
            norms   = [torch.sum(w**2) for w in weights]

            for i, p in enumerate(weights):
                if i == len(weights) - 1:
                    # At the final layer, we enforce L2 on weights only
                    # L   = alpha * ||W_i||^2
                    # dL  = 2 * alpha * W_i
                    # upd = W += (-lr * 2 * alpha) * ||W_i||
                    prior_grad  = p
                    prior_alpha = -lr * 2.0 * alpha
                    p.add_(prior_grad, alpha=prior_alpha)

                else:
                    # At every layer before the last, we enforce balance only.
                    # L   = beta * (||W_i||^2 - ||W_{i+1}||^2)^2
                    # dL  = 4 * beta * (||W_i||^2 - ||W_{i+1}||^2) * W_i
                    # upd = W += (-lr * 4 * beta * (||W_i||^2 - ||W_{i+1}||^2))
                    diff = norms[i] - norms[i + 1]

                    prior_grad  = p
                    prior_alpha = -lr * 4.0 * beta * diff
                    p.add_(prior_grad, alpha=prior_alpha)

        return loss


### Helpers

In [ ]:
def get_network_balance(model, per_layer = False):
    with torch.no_grad():
        weights = [p for p in model.parameters() if len(p.shape) > 1]
        norms   = [torch.sum(w**2) for w in weights]
        # TODO: Explore len(p.shape) <= 1 parameters

        diffs   = [(norms[i] - norms[i+1])**2 for i in range(len(norms) - 1)]
        balance = torch.stack(diffs)
        
        if per_layer:
            return balance.tolist()
        else:
            return torch.mean(balance).item()

In [ ]:
def get_dataLoader(dataset_type, train, batch_size):
    match dataset_type:
        case "MNIST":
            return DataLoader(
                datasets.MNIST('./data', train=train, download=True, transform=mnist_transform), 
                batch_size=batch_size, 
                shuffle=True,
                pin_memory=True,
            )
            
        case "CIFAR10":
            return DataLoader(
                datasets.CIFAR10('./data', train=train, download=True, transform=cifar10_transform), 
                batch_size=batch_size, 
                shuffle=True,
                pin_memory=True,
            )
            
        case "CIFAR100":
            return DataLoader(
                datasets.CIFAR100('./data', train=train, download=True, transform=cifar100_transform), 
                batch_size=batch_size, 
                shuffle=True,
                pin_memory=True,
            )
            
        case _:
            raise ValueError(f"'{dataset_type}' not defined as a dataset. ")

In [ ]:
def get_optimizer(
    optimizer_name: str,
    params,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    momentum: float = 0.9,
    alpha: float = 0.1,
    beta: float = 0.1
):
    match optimizer_name:
        case "adamw":
            return torch.optim.AdamW(
                params,
                lr=lr,
                weight_decay=weight_decay,
            )

        case "sgd":
            return torch.optim.SGD(
                params,
                lr=lr,
                weight_decay=weight_decay,
                momentum=momentum,
            )

        case "rsb":
            return RSB(
                params,
                lr=lr,
                alpha=alpha,
                beta=beta,
            )

        case _:
            raise ValueError(f"'{optimizer_name}' not defined as an optimizer. ")

### Training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[device]: ", device)

In [ ]:
def train(config=None):
    with wandb.init(project=PROJECT_NAME, config=default_config):
        config = wandb.config

        wandb.run.name = (
            f"lr={config.lr:.0e}_"
            f"bs={config.batch_size}_"
            f"a={config.alpha}_b={config.beta}"
        )

        # MNIST is 28x28
        model = CNN(input_size=(28, 28)).to(device)
        criterion = RSB_Loss(alpha=config.alpha, beta=config.beta)
        train_loader = get_dataLoader(
            dataset_type=config.dataset,
            train=True,
            batch_size=config.batch_size,
        )
        optimizer = get_optimizer(
            optimizer_name=config.optimizer,
            params=model.parameters(),
            lr=config.lr,
            weight_decay=config.get("weight_decay", 0.0),
            momentum=config.get("momentum", 0.9),
            alpha=config.alpha,
            beta=config.beta,
        )

        for epoch in range(config.epochs):
            model.train()
            running_loss = 0.0
            for batch_idx, (data, target) in enumerate(train_loader):
                data, target = data.to(device), target.to(device)

                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target, model)
                loss.backward()
                optimizer.step()

                running_loss += loss.item()
            avg_loss = running_loss / len(train_loader)
            wandb.log({"epoch": epoch, "train_loss": avg_loss})

In [ ]:
train()